# Case B — Build: Liquor Census x Calendar Events (Monthly)

**Purpose:** a *build* notebook, in the same spirit as `caseb_build_data.ipynb` --
it doesn't model anything, it just produces a clean, merged dataset that a
downstream analysis notebook can load and start from. This one combines
`liquor_census.parquet` with the same calendar-events table used in
`caseb_calendar_events.ipynb` (imported the same way -- fixed-rule holidays
computed, three occasions looked up directly: Super Bowl Sunday, the Iowa
State Fair, and Hawkeyes/Cyclones home games).

**Read section 1 before using the output.** `liquor_census.parquet` is a
fundamentally different shape of data from `liquor_2022_2026.parquet` (the
file `caseb_calendar_events.ipynb` uses), and that difference breaks the
"week + month-fixed-effects + event dummies" regression setup from that
notebook if copied here unchanged. This notebook flags exactly why, with a
numeric check, so whoever builds the monthly analysis notebook doesn't
re-discover it the hard way.

In [3]:
import duckdb as db
import numpy as np
import pandas as pd

con = db.connect()

## 1. What shape is `liquor_census.parquet`, actually?

`caseb_calendar_events.ipynb` starts from **daily, store-level** wholesale
line items (`ordered_on`, `store_city`, `sales_bottles`, `sales_dollars`)
and rolls them up to weekly totals itself. `liquor_census.parquet` is a
different animal: check the grain before assuming it works the same way.

In [4]:
census = con.execute("SELECT * FROM 'liquor_census.parquet'").df()

print(f"{len(census):,} rows, {census['category_name'].nunique()} categories, "
      f"{census['year_month'].nunique()} months "
      f"({census['year_month'].min()} to {census['year_month'].max()})")
print("columns:", list(census.columns))
census.head()

2,474 rows, 52 categories, 56 months (2022-01 to 2026-08)
columns: ['category_name', 'year_month', 'year', 'month_num', 'total_units', 'total_dollars', 'total_liters', 'avg_bottle_price', 'realized_price_per_unit', 'n_stores', 'n_zips_reached', 'n_transactions', 'log_units', 'log_dollars', 'log_liters']


,category_name,year_month,year,month_num,total_units,total_dollars,total_liters,avg_bottle_price,realized_price_per_unit,n_stores,n_zips_reached,n_transactions,log_units,log_dollars,log_liters
0,100% AGAVE TEQUILA,2022-01,2022,1,50549.0,1443274.89,34750.75,NaN,28.551997,885,271,5596,10.830718,14.182426,10.455985
1,AGED DARK RUM,2022-01,2022,1,3840.0,79755.80,3004.00,NaN,20.769740,245,134,674,8.253488,11.286737,8.008033
2,AMERICAN BRANDIES,2022-01,2022,1,61714.0,407451.14,32754.75,NaN,6.602248,928,303,5407,11.030282,12.917679,10.396834
3,AMERICAN CORDIALS & LIQUEURS,2022-01,2022,1,35966.0,282952.50,17391.38,NaN,7.867222,914,314,4327,10.490357,12.553038,9.763787
4,AMERICAN DISTILLED SPIRITS SPECIALTY,2022-01,2022,1,4903.0,88699.48,3635.35,NaN,18.090859,368,204,736,8.497806,11.393021,8.198736


**This file is pre-aggregated to category x month, statewide.** There is
no `store_city`, no daily date, no individual transaction -- just monthly
totals per liquor category (`total_units`, `total_dollars`, `total_liters`,
plus reach metrics `n_stores` / `n_zips_reached` / `n_transactions` and
already-computed `avg_bottle_price`, `realized_price_per_unit`, and
`log_*` columns). Two consequences up front:

- **No Iowa City / Ames split is possible with this file.** The Hawkeyes /
  Cyclones home-game analysis from `caseb_calendar_events.ipynb` can't be
  reproduced here -- there's no geography column to filter on. Home-game
  flags are still built below (for consistency with the source notebook's
  event list, and in case a statewide effect is genuinely of interest), but
  they're diluted across all of Iowa, not isolated to the two college
  towns where the effect actually showed up.
- **The finest available time grain is the calendar month**, not the week.
  That's a big drop in resolution -- 56 months vs. roughly 240 weeks -- and,
  as section 4 shows, it's not just "less resolution," it breaks something
  structurally.

## 2. Collapse to a statewide monthly total (alongside the category-level detail)

Two useful shapes to carry forward: the full category-level long table
(for anyone who wants "was July 4th better for tequila specifically"), and
a statewide total across all categories per month (the direct analog of
`weekly_state` in the source notebook).

In [5]:
monthly_total = census.groupby(["year_month", "year", "month_num"], as_index=False).agg(
    total_units=("total_units", "sum"),
    total_dollars=("total_dollars", "sum"),
    total_liters=("total_liters", "sum"),
    n_stores=("n_stores", "sum"),
    n_transactions=("n_transactions", "sum"),
).sort_values("year_month").reset_index(drop=True)

monthly_total["log_units"] = np.log1p(monthly_total["total_units"])
monthly_total["log_dollars"] = np.log1p(monthly_total["total_dollars"])

print(f"{len(monthly_total)} months, {monthly_total['year_month'].min()} to {monthly_total['year_month'].max()}")
monthly_total.head()

56 months, 2022-01 to 2026-08


,year_month,year,month_num,total_units,total_dollars,total_liters,n_stores,n_transactions,log_units,log_dollars
0,2022-01,2022,1,2062829.0,27597708.30,1567197.99,32645,182595,14.539589,17.133243
1,2022-02,2022,2,2094442.0,29474100.93,1664280.51,31959,174284,14.554798,17.199023
2,2022-03,2022,3,2190674.0,30225874.55,1728013.63,32583,185496,14.599720,17.224209
3,2022-04,2022,4,2560073.0,34781761.59,1987119.31,35353,211131,14.755547,17.364604
4,2022-05,2022,5,2610732.0,35948167.34,2059474.94,35199,217820,14.775142,17.397589


## 3. Build the calendar-events table

Same importer as `caseb_calendar_events.ipynb`, unchanged: `nth_weekday` /
`last_weekday` compute the fixed-rule holidays, and Super Bowl Sunday,
the Iowa State Fair, and Hawkeyes/Cyclones home dates are looked up
directly (see that notebook for sources). The only change is the grain
`events_df` gets rolled up to -- `year_month` instead of `week` -- since
that's what this census file can join on.

**Not included, same as the source notebook:** RAGBRAI (route changes
towns every year).

In [6]:
def nth_weekday(year, month, weekday, n):
    """nth occurrence of `weekday` (Mon=0..Sun=6) in a given month."""
    first = pd.Timestamp(year=year, month=month, day=1)
    return first + pd.Timedelta(days=(weekday - first.weekday()) % 7 + 7 * (n - 1))


def last_weekday(year, month, weekday):
    """last occurrence of `weekday` in a given month."""
    month_end = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    return month_end - pd.Timedelta(days=(month_end.weekday() - weekday) % 7)


events = {}


def add_event(date, name):
    events.setdefault(pd.Timestamp(date), []).append(name)


# looked up directly (not computable from a fixed rule) -- see caseb_calendar_events.ipynb for sources
SUPER_BOWL_SUNDAY = {
    2022: "2022-02-13", 2023: "2023-02-12", 2024: "2024-02-11",
    2025: "2025-02-09", 2026: "2026-02-08",
}
IOWA_STATE_FAIR = {
    2022: ("2022-08-11", "2022-08-21"), 2023: ("2023-08-10", "2023-08-20"),
    2024: ("2024-08-08", "2024-08-18"), 2025: ("2025-08-07", "2025-08-17"),
    2026: ("2026-08-13", "2026-08-23"),
}
HAWKEYES_HOME = {
    2022: ["2022-09-03", "2022-09-10", "2022-09-17", "2022-10-01", "2022-10-29", "2022-11-12", "2022-11-25"],
    2023: ["2023-09-02", "2023-09-16", "2023-09-30", "2023-10-07", "2023-10-21", "2023-11-11", "2023-11-18"],
    2024: ["2024-08-31", "2024-09-07", "2024-09-14", "2024-10-12", "2024-10-26", "2024-11-02", "2024-11-29"],
    2025: ["2025-08-30", "2025-09-13", "2025-09-27", "2025-10-18", "2025-10-25", "2025-11-08", "2025-11-22"],
    2026: ["2026-09-05"],
}
CYCLONES_HOME = {
    2022: ["2022-09-03", "2022-09-17", "2022-09-24", "2022-10-08", "2022-10-29", "2022-11-05", "2022-11-19"],
    2023: ["2023-09-02", "2023-09-09", "2023-09-23", "2023-10-07", "2023-11-04", "2023-11-18"],
    2024: ["2024-08-31", "2024-09-21", "2024-10-05", "2024-10-19", "2024-11-02", "2024-11-09", "2024-11-16"],
    2025: ["2025-08-30", "2025-09-06", "2025-09-27", "2025-10-25", "2025-11-01", "2025-11-22"],
    2026: ["2026-09-05", "2026-09-12"],
}

for year in range(2022, 2027):
    add_event(f"{year}-01-01", "New Year's Day")
    add_event(f"{year}-12-31", "New Year's Eve")
    add_event(SUPER_BOWL_SUNDAY[year], "Super Bowl Sunday")
    add_event(f"{year}-03-17", "St. Patrick's Day")
    add_event(f"{year}-05-05", "Cinco de Mayo")
    add_event(last_weekday(year, 5, 0), "Memorial Day")
    add_event(f"{year}-07-04", "July 4th")
    add_event(nth_weekday(year, 9, 0, 1), "Labor Day")
    add_event(f"{year}-10-31", "Halloween")
    thanksgiving = nth_weekday(year, 11, 3, 4)
    add_event(thanksgiving, "Thanksgiving")
    add_event(thanksgiving - pd.Timedelta(days=1), "Thanksgiving")
    add_event(f"{year}-12-24", "Christmas")
    add_event(f"{year}-12-25", "Christmas")

    fair_start, fair_end = IOWA_STATE_FAIR[year]
    for day in pd.date_range(fair_start, fair_end):
        add_event(day, "Iowa State Fair")

for date in [d for season in HAWKEYES_HOME.values() for d in season]:
    add_event(date, "Hawkeyes Home Game")
for date in [d for season in CYCLONES_HOME.values() for d in season]:
    add_event(date, "Cyclones Home Game")

events_df = pd.DataFrame(
    [(date, names) for date, names in sorted(events.items())], columns=["date", "events"]
)
events_df["year_month"] = events_df["date"].dt.strftime("%Y-%m")

EVENT_TYPES = [
    "New Year's Day", "New Year's Eve", "Super Bowl Sunday", "St. Patrick's Day",
    "Cinco de Mayo", "Memorial Day", "July 4th", "Labor Day", "Halloween",
    "Thanksgiving", "Christmas", "Iowa State Fair", "Hawkeyes Home Game", "Cyclones Home Game",
]

print(f"{len(events_df)} event-dates built, {events_df['year_month'].nunique()} distinct months touched")
events_df.head(10)

164 event-dates built, 50 distinct months touched


,date,events,year_month
0,2022-01-01,[New Year's Day],2022-01
1,2022-02-13,[Super Bowl Sunday],2022-02
2,2022-03-17,[St. Patrick's Day],2022-03
3,2022-05-05,[Cinco de Mayo],2022-05
4,2022-05-30,[Memorial Day],2022-05
5,2022-07-04,[July 4th],2022-07
6,2022-08-11,[Iowa State Fair],2022-08
7,2022-08-12,[Iowa State Fair],2022-08
8,2022-08-13,[Iowa State Fair],2022-08
9,2022-08-14,[Iowa State Fair],2022-08


## 4. Attach event flags -- and check for the identification problem this grain creates

At weekly grain, a fixed-rule holiday lands in a *different week number*
most years (a "3rd Monday" drifts around the calendar), which is what lets
`caseb_calendar_events.ipynb` separate "this week has an event" from "this
is generically a September week" using month fixed effects. At **monthly**
grain that separation disappears: every one of these events falls in the
*same calendar month every single year*, with zero exceptions. The cell
below builds both a presence dummy and a same-month day-count per event
(the day-count carries more information at this grain -- an 11-day Iowa
State Fair is not the same as a 1-day holiday), then proves the
collinearity numerically rather than asserting it.

In [7]:
def build_month_flags(df):
    out = df.copy()
    for event_type in EVENT_TYPES:
        mask = events_df["events"].apply(lambda lst: event_type in lst)
        day_counts = events_df.loc[mask].groupby("year_month").size()
        slug = event_type.lower().replace(" ", "_").replace(".", "").replace("'", "")
        out[f"{slug}_days"] = out["year_month"].map(day_counts).fillna(0).astype(int)
        out[f"{slug}_present"] = (out[f"{slug}_days"] > 0).astype(int)

    present_cols = [c for c in out.columns if c.endswith("_present")]
    out["n_event_types_in_month"] = out[present_cols].sum(axis=1)
    return out


monthly_total_f = build_month_flags(monthly_total)
census_f = census.merge(
    monthly_total_f[["year_month"] + [c for c in monthly_total_f.columns
                                       if c.endswith(("_present", "_days")) or c == "n_event_types_in_month"]],
    on="year_month", how="left",
)

print(f"monthly_total_f: {monthly_total_f.shape}, census_f: {census_f.shape}")
monthly_total_f[["year_month", "new_years_day_present", "super_bowl_sunday_present",
                  "iowa_state_fair_days", "n_event_types_in_month"]].head(12)

monthly_total_f: (56, 39), census_f: (2474, 44)


,year_month,new_years_day_present,super_bowl_sunday_present,iowa_state_fair_days,n_event_types_in_month
0,2022-01,1,0,0,1
1,2022-02,0,1,0,1
2,2022-03,0,0,0,1
3,2022-04,0,0,0,0
4,2022-05,0,0,0,2
5,2022-06,0,0,0,0
6,2022-07,0,0,0,1
7,2022-08,0,0,11,1
8,2022-09,0,0,0,3
9,2022-10,0,0,0,3


In [8]:
# the collinearity check: cross-tabulate month-of-year against each event's presence dummy.
# a 0/1 perfectly aligned with a single month value means the event dummy IS a month-of-year
# dummy in disguise -- OLS can't tell them apart, so including both is redundant, and dropping
# month fixed effects to keep the event dummies just relabels "month effect" as "event effect."
for event_type in EVENT_TYPES:
    slug = event_type.lower().replace(" ", "_").replace(".", "").replace("'", "")
    months_touched = sorted(monthly_total_f.loc[monthly_total_f[f"{slug}_present"] == 1, "month_num"].unique())
    print(f"{event_type:22s} -> always month(s) {months_touched}")

New Year's Day         -> always month(s) [np.int64(1)]
New Year's Eve         -> always month(s) [np.int64(12)]
Super Bowl Sunday      -> always month(s) [np.int64(2)]
St. Patrick's Day      -> always month(s) [np.int64(3)]
Cinco de Mayo          -> always month(s) [np.int64(5)]
Memorial Day           -> always month(s) [np.int64(5)]
July 4th               -> always month(s) [np.int64(7)]
Labor Day              -> always month(s) [np.int64(9)]
Halloween              -> always month(s) [np.int64(10)]
Thanksgiving           -> always month(s) [np.int64(11)]
Christmas              -> always month(s) [np.int64(12)]
Iowa State Fair        -> always month(s) [np.int64(8)]
Hawkeyes Home Game     -> always month(s) [np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Cyclones Home Game     -> always month(s) [np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


Every fixed-rule holiday, Super Bowl Sunday, and the Iowa State Fair maps
to **exactly one calendar month, every year, no exceptions**. Only the
Hawkeyes/Cyclones home-game flags spread across more than one month
(Aug-Nov, following the football schedule) -- and even those overlap
months already claimed by Labor Day, Halloween, and Thanksgiving.

**What this means for a downstream analysis notebook:** don't copy the
`month fixed effects + event dummies` regression from `caseb_calendar_events.ipynb`
onto this monthly data unchanged -- `pd.get_dummies(df, columns=["month_num"])`
and most of the event-presence dummies above would be near-perfect linear
combinations of each other, and `statsmodels` will either drop columns
automatically or return coefficients with enormous standard errors that
look like "no effect" but are really "the model can't tell these two
dummies apart." Two ways around it if a monthly analysis is still useful:
drop the calendar-month fixed effects and treat the event dummy *as* the
month effect (accept that "July 4th's coefficient" and "being July" are
the same number), or lean on the `_days` count columns and
`n_event_types_in_month` as continuous regressors instead of one dummy
per event, which at least lets multi-event months (Sept, Dec) be
distinguished from single-event ones without needing month fixed effects
at all.

## 5. Save the combined outputs

Two files, matching the two shapes built above:

- **`liquor_census_monthly_events.parquet`** -- category-level detail
  (52 categories x 56 months) with event flags attached, for anyone who
  wants a category-specific cut.
- **`liquor_census_statewide_monthly_events.parquet`** -- the collapsed
  statewide monthly total with event flags, the direct analog of
  `weekly_state` from the source notebook, ready for a regression (with
  the section-4 caveat in mind).

In [9]:
con.execute("COPY census_f TO 'liquor_census_monthly_events.parquet' (FORMAT PARQUET)")
con.execute("COPY monthly_total_f TO 'liquor_census_statewide_monthly_events.parquet' (FORMAT PARQUET)")

print("wrote liquor_census_monthly_events.parquet:", census_f.shape)
print("wrote liquor_census_statewide_monthly_events.parquet:", monthly_total_f.shape)

wrote liquor_census_monthly_events.parquet: (2474, 44)
wrote liquor_census_statewide_monthly_events.parquet: (56, 39)
